# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [1]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [2]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/billpayment"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 5

--- BillPayment_20260207170422.parquet ---
  Registros: 2
  Colunas: ['raw_value']
  ❌ Não contém dados JSON

--- BillPayment_20260207170543.parquet ---
  Registros: 2
  Colunas: ['raw_value']
  ❌ Não contém dados JSON

--- BillPayment_20260325162037.parquet ---
  Registros: 10
  Colunas: ['VendorRef', 'PayType', 'CreditCardPayment', 'TotalAmt', 'domain', 'sparse', 'Id', 'SyncToken', 'MetaData', 'DocNumber', 'TxnDate', 'CurrencyRef', 'Line', 'CheckPayment']
  ✅ Contém JSON nas colunas: ['VendorRef', 'CreditCardPayment', 'MetaData', 'CurrencyRef', 'Line', 'CheckPayment']

--- BillPayment_20260325213744.parquet ---
  Registros: 10
  Colunas: ['VendorRef', 'PayType', 'CreditCardPayment', 'TotalAmt', 'domain', 'sparse', 'Id', 'SyncToken', 'MetaData', 'DocNumber', 'TxnDate', 'CurrencyRef', 'Line', 'CheckPayment']
  ✅ Contém JSON nas colunas: ['VendorRef', 'CreditCardPayment', 'MetaData', 'CurrencyRef', 'Line', 'CheckPayment']

--- BillPayment_20260325214126.parquet 

import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [3]:
from pyspark.sql.functions import from_json, schema_of_json, col, explode
from pyspark.sql.types import ArrayType, StructType

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema))

        # Se o resultado for um array, explodir primeiro
        parsed_type = df_temp.schema["parsed"].dataType
        if isinstance(parsed_type, ArrayType):
            df_temp = df_temp.withColumn("parsed", explode(col("parsed")))

        # Manter colunas originais (sem a coluna JSON bruta) + colunas expandidas do JSON
        colunas_originais = [c for c in df.columns if c != col_json]
        df_temp = df_temp.select(*colunas_originais, col("parsed.*"))

        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 3

Comparando: BillPayment_20260325162037.parquet vs BillPayment_20260325213744.parquet
  ✅ Mesmas colunas (15 colunas)
  ✅ Tipos de dados idênticos

Comparando: BillPayment_20260325162037.parquet vs BillPayment_20260325214126.parquet
  ✅ Mesmas colunas (15 colunas)
  ✅ Tipos de dados idênticos

✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!


In [4]:
from pyspark.sql.functions import from_json, schema_of_json, col, explode
from pyspark.sql.types import StructType, StringType, ArrayType
from functools import reduce

def simplify_value_only_columns(df):
    """Se uma coluna for um struct com apenas a chave 'value', substitui pelo valor direto."""
    for field in df.schema.fields:
        if isinstance(field.dataType, StructType):
            nomes_campos = [f.name for f in field.dataType.fields]
            if nomes_campos == ["value"]:
                df = df.withColumn(field.name, col(f"{field.name}.value"))
    return df

def flatten_df(df, sep="_"):
    """Achata recursivamente colunas StructType e explode ArrayType."""
    df = simplify_value_only_columns(df)
    changed = True
    while changed:
        changed = False

        # Explodir colunas array
        for field in df.schema.fields:
            if isinstance(field.dataType, ArrayType):
                df = df.withColumn(field.name, explode(col(field.name)))
                changed = True

        # Achatar colunas struct
        novas_colunas = []
        for field in df.schema.fields:
            if isinstance(field.dataType, StructType):
                nomes_campos = [f.name for f in field.dataType.fields]
                if nomes_campos == ["value"]:
                    novas_colunas.append(df[field.name]["value"].alias(field.name))
                else:
                    changed = True
                    for sub in field.dataType.fields:
                        novo_nome = f"{field.name}{sep}{sub.name}"
                        novas_colunas.append(df[field.name][sub.name].alias(novo_nome))
            else:
                novas_colunas.append(df[field.name])
        df = df.select(novas_colunas)
        df = simplify_value_only_columns(df)
    return df

def parse_nested_json(df, sep="_", max_depth=5):
    """Detecta e parseia colunas string que contêm JSON, depois achata."""
    for _ in range(max_depth):
        str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        if not str_cols:
            break

        # Coletar amostras não-nulas de cada coluna string
        cols_json = []
        for c in str_cols:
            row = df.select(c).filter(col(c).isNotNull()).first()
            if row:
                val = row[0]
                if val and isinstance(val, str) and val.strip() and val.strip()[0] in ('{', '['):
                    cols_json.append((c, val))

        if not cols_json:
            break

        for c, val in cols_json:
            schema = schema_of_json(val)
            df = df.withColumn(c, from_json(df[c], schema))

        df = flatten_df(df, sep)
    return df

# Processar cada arquivo: parsear TODAS as colunas JSON em sequência no mesmo DataFrame
dfs_processados = []

for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"⚠️ Pulando {nome}: sem dados JSON para processar ({info['df'].columns})")
        continue

    df = info["df"]

    # Parsear todas as colunas JSON de uma vez, em sequência no mesmo df
    for col_json in info["colunas_json"]:
        amostra_row = df.select(col_json).filter(col(col_json).isNotNull()).first()
        if not amostra_row:
            continue
        amostra = amostra_row[0]
        json_schema = schema_of_json(amostra)

        # Substituir a coluna JSON pela versão parseada
        df = df.withColumn(col_json, from_json(col(col_json), json_schema))

    # Achatar tudo (structs, arrays, value-only) de uma vez
    df = flatten_df(df)

    # Parsear sub-JSONs que possam ter surgido como strings após achatamento
    df = parse_nested_json(df)
    df = simplify_value_only_columns(df)

    info["df_parsed"] = df
    dfs_processados.append(df)
    print(f"✅ {nome}: {df.count()} registros, {len(df.columns)} colunas")

# Unir todos os DataFrames e remover duplicados
df_final = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_processados)
df_final = df_final.dropDuplicates()

print(f"\nTotal de registros (sem duplicados): {df_final.count()}")
print(f"Total de colunas: {len(df_final.columns)}")
df_final.printSchema()

⚠️ Pulando BillPayment_20260207170422.parquet: sem dados JSON para processar (['raw_value'])
⚠️ Pulando BillPayment_20260207170543.parquet: sem dados JSON para processar (['raw_value'])
✅ BillPayment_20260325162037.parquet: 10 registros, 22 colunas
✅ BillPayment_20260325213744.parquet: 10 registros, 22 colunas
✅ BillPayment_20260325214126.parquet: 10 registros, 22 colunas

Total de registros (sem duplicados): 10
Total de colunas: 22
root
 |-- VendorRef_name: string (nullable = true)
 |-- VendorRef_value: string (nullable = true)
 |-- PayType: string (nullable = true)
 |-- CreditCardPayment_CCAccountRef_name: string (nullable = true)
 |-- CreditCardPayment_CCAccountRef_value: string (nullable = true)
 |-- TotalAmt: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- sparse: string (nullable = true)
 |-- Id: string (nullable = true)
 |-- SyncToken: string (nullable = true)
 |-- MetaData_CreateTime: string (nullable = true)
 |-- MetaData_LastUpdatedTime: string (nullable =

In [5]:
from IPython.display import display

# Mostrar tabela única consolidada
display(df_final.toPandas())

,VendorRef_name,VendorRef_value,PayType,CreditCardPayment_CCAccountRef_name,CreditCardPayment_CCAccountRef_value,TotalAmt,domain,sparse,Id,SyncToken,...,DocNumber,TxnDate,CurrencyRef_name,CurrencyRef_value,Line_Amount,Line_LinkedTxn_TxnId,Line_LinkedTxn_TxnType,CheckPayment_BankAccountRef_name,CheckPayment_BankAccountRef_value,CheckPayment_PrintStatus
0,Robertson & Associates,49,Check,None,None,300.0,QBO,False,91,0,...,10,2025-08-25,United States Dollar,USD,300.00,90,Bill,Checking,35,NotSet
1,PG&E,48,Check,None,None,114.09,QBO,False,77,0,...,6,2025-11-26,United States Dollar,USD,114.09,35,Bill,Checking,35,NotSet
2,Books by Bessie,30,Check,None,None,75.0,QBO,False,37,0,...,3,2025-11-25,United States Dollar,USD,75.00,18,Bill,Checking,35,NotSet
3,Cal Telephone,32,CreditCard,Mastercard,41,56.5,QBO,False,117,0,...,1,2025-11-27,United States Dollar,USD,56.50,105,Bill,None,None,None
4,Tim Philip Masonry,51,Check,None,None,666.0,QBO,False,82,0,...,45,2025-11-26,United States Dollar,USD,666.00,3,Bill,Checking,35,NotSet
5,Cal Telephone,32,CreditCard,Mastercard,41,74.36,QBO,False,50,0,...,1,2025-11-25,United States Dollar,USD,74.36,21,Bill,None,None,None
6,Norton Lumber and Building Materials,46,CreditCard,Mastercard,41,103.55,QBO,False,118,0,...,1,2025-11-27,United States Dollar,USD,103.55,25,Bill,None,None,None
7,Hicks Hardware,41,Check,None,None,250.0,QBO,False,79,0,...,7,2025-11-06,United States Dollar,USD,250.00,2,Bill,Checking,35,NotSet
8,Brosnahan Insurance Agency,31,Check,None,None,2000.0,QBO,False,22,0,...,1,2025-11-24,United States Dollar,USD,2000.00,1,Bill,Checking,35,NotSet
9,Hall Properties,40,Check,None,None,900.0,QBO,False,104,0,...,11,2025-11-20,United States Dollar,USD,900.00,24,Bill,Checking,35,NotSet


In [6]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
